In [1]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Data Load karna
df = sns.load_dataset('titanic')

# Kaam ke columns rakh rahe hain aur baaki drop kar rahe hain (jaise 'deck', 'alive' jo duplicate data hain)
columns_to_keep = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[columns_to_keep]

print("--- Data Cleaning Se Pehle Missing Values ---")
print(df.isnull().sum())

# Step 2: Key Challenge - Data Cleaning (Missing Values)
# 1. Age me bohot missing values hain, unhe hum data ke Median (beech ki value) se bhar dete hain.
df['age'] = df['age'].fillna(df['age'].median())

# 2. Embarked (kahan se baithe) me sirf 2 values missing hain, use sabse zyada baar aane wali value (Mode) se bhar dete hain.
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Step 3: Text Columns ko Numeric me badalna (Encoding)
# 'sex' aur 'embarked' text hain, inhen 0 aur 1 me convert karenge.
df_encoded = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True)

# Step 4: Features (X) aur Target (y) alag karna
X = df_encoded.drop('survived', axis=1)
y = df_encoded['survived']

# Data ko Train aur Test me split karna (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 5: Stacking Framework Khada Karna
# Hum do alag-alag dimaag (Models) use kar rahe hain jo data ko alag tarike se dekhte hain.
base_models = [
    ('rf_clf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb_clf', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

# Boss Layer (Meta-Model): Yeh classifier hai, toh Logistic Regression sabse best kaam karta hai predictions ko jodne ke liye.
boss_model = LogisticRegression()

# Final Stacking Classifier banana
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=boss_model,
    cv=5 # Internal cross-validation taaki overfitting na ho
)

# Model ko train karna
stacking_clf.fit(X_train, y_train)

# Step 6: Check karna ki Boss Layer ne kis model ko kitna weight diya?
# Classification me hum coefficients (.coef_) nikal kar dekhte hain ki kis base model ke prediction ko boss ne zyada importance di.
weights = stacking_clf.final_estimator_.coef_[0]
print("\n--- Boss Layer (Meta-Model) Ka Insaaf ---")
print(f"Random Forest (rf_clf) ko mila weightage coefficient: {weights[0]:.4f}")
print(f"Gradient Boosting (gb_clf) ko mila weightage coefficient: {weights[1]:.4f}")

# Step 7: Final Accuracy Check karna
y_pred = stacking_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\n--- Stacking Model Final Results ---")
print(f"Overall Accuracy: {accuracy * 100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred))

--- Data Cleaning Se Pehle Missing Values ---
survived      0
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
dtype: int64

--- Boss Layer (Meta-Model) Ka Insaaf ---
Random Forest (rf_clf) ko mila weightage coefficient: 1.7224
Gradient Boosting (gb_clf) ko mila weightage coefficient: 3.0797

--- Stacking Model Final Results ---
Overall Accuracy: 82.68%

Detailed Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86       105
           1       0.82      0.74      0.78        74

    accuracy                           0.83       179
   macro avg       0.83      0.81      0.82       179
weighted avg       0.83      0.83      0.83       179



# Using Imputer

In [3]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

# ==========================================
# 1. DATA LOADING AND CLEANING
# ==========================================
df = sns.load_dataset('titanic')

# Sirf kaam ke columns rakh rahe hain, baaki sab hata diya
kaam_ke_columns = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[kaam_ke_columns]

# Embarked ki khali jagah ko most frequent city se fill kar rahe hain
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Text columns ko numbers me convert kar rahe hain (One-Hot Encoding)
df_numeric = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True)


# ==========================================
# 2. KNN IMPUTER FOR MISSING AGE (Aapka Solid Logic)
# ==========================================
# 'survived' ko chhod kar baaki saare features se age ka andaza lagayenge
features = df_numeric.drop('survived', axis=1)

# n_neighbors=3 matlab 3 milti-julti sawariyon ka average nikalega
imputer = KNNImputer(n_neighbors=3)
features_clean = imputer.fit_transform(features)

# Matrix ko wapas DataFrame (table) me convert kar rahe hain
X = pd.DataFrame(features_clean, columns=features.columns)
y = df_numeric['survived']


# ==========================================
# 3. TRAIN-TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# ==========================================
# 4. STACKING CONFIGURATION (Multi-Model Setup)
# ==========================================
# Hamare do alag-alag base models (Dimaag 1 aur Dimaag 2)
dimaag_1 = RandomForestClassifier(n_estimators=100, random_state=42)
dimaag_2 = GradientBoostingClassifier(n_estimators=100, random_state=42)

# Final decision lene ke liye Boss Layer bitha di
boss_model = LogisticRegression()

# Stacking model taiyar kar rahe hain jisme dono base models aur boss layer hain
final_stack_model = StackingClassifier(
    estimators=[('rf', dimaag_1), ('gb', dimaag_2)],
    final_estimator=boss_model,
    cv=5
)

# Model ko train kar rahe hain
final_stack_model.fit(X_train, y_train)


# ==========================================
# 5. FINAL ACCURACY CHECK
# ==========================================
accuracy = final_stack_model.score(X_test, y_test)

print("*" * 50)
print(f"Final Accuracy Aayi Hai: {accuracy * 100:.2f}%")
print("*" * 50)

**************************************************
Final Accuracy Aayi Hai: 82.68%
**************************************************
